In [22]:
%load_ext autoreload
%autoreload 2

In [23]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

## Deep dive into our dataset.

When analysing the dataset from `FibonacciModDataset` in our `model_dim_4_layer_3.py` we noticed that loss for evulate set is less than the loss for training set, which is quite the opposite.

In [24]:
from script.model_dim_4_layer_3 import FibonacciModDataset
from torch.utils.data import Dataset, DataLoader, random_split


seq_len = 20
batch_size = 6
generated_ds = FibonacciModDataset(num_samples=25000, mod=vocab_size, seq_len=seq_len)

train_size = int(0.8 * len(generated_ds))
test_size = len(generated_ds) - train_size
train_ds, test_ds = random_split(generated_ds, [train_size, test_size])

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)
test_loader = DataLoader(test_ds, batch_size=batch_size, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)

c = 0

for x, y in train_ds:
    if c >= 5:
        break
    print(x, y)

    c += 1



tensor([3, 5, 8, 3, 1, 4, 5, 9, 4, 3, 7, 0, 7, 7, 4, 1, 5, 6, 1, 7]) tensor([5, 8, 3, 1, 4, 5, 9, 4, 3, 7, 0, 7, 7, 4, 1, 5, 6, 1, 7, 8])
tensor([1, 2, 3, 5, 8, 3, 1, 4, 5, 9, 4, 3, 7, 0, 7, 7, 4, 1, 5, 6]) tensor([2, 3, 5, 8, 3, 1, 4, 5, 9, 4, 3, 7, 0, 7, 7, 4, 1, 5, 6, 1])
tensor([9, 0, 9, 9, 8, 7, 5, 2, 7, 9, 6, 5, 1, 6, 7, 3, 0, 3, 3, 6]) tensor([0, 9, 9, 8, 7, 5, 2, 7, 9, 6, 5, 1, 6, 7, 3, 0, 3, 3, 6, 9])
tensor([9, 1, 0, 1, 1, 2, 3, 5, 8, 3, 1, 4, 5, 9, 4, 3, 7, 0, 7, 7]) tensor([1, 0, 1, 1, 2, 3, 5, 8, 3, 1, 4, 5, 9, 4, 3, 7, 0, 7, 7, 4])
tensor([1, 5, 6, 1, 7, 8, 5, 3, 8, 1, 9, 0, 9, 9, 8, 7, 5, 2, 7, 9]) tensor([5, 6, 1, 7, 8, 5, 3, 8, 1, 9, 0, 9, 9, 8, 7, 5, 2, 7, 9, 6])



Basically we are generating random shamples and then using sliding window to generate shamples from those.
Now the randomness is in the start_idx, we want to see the dublicates pair to better understand it.
So we want to see the dublicates in both train and test set pais because, if the model has already seen
the data then it will try to find a shortcut

In [53]:
import torch

print(len(train_ds))
print(len(test_ds))
print(len(train_ds) + len(test_ds)) # that should correspond to the shamples provided.

pairs = []
for x, y in train_ds:
    x = torch.cat((x, y[-1: ]))
    pairs.append(x)
print(pairs[0: 4])


20000
5000
25000
[tensor([3, 5, 8, 3, 1, 4, 5, 9, 4, 3, 7, 0, 7, 7, 4, 1, 5, 6, 1, 7, 8]), tensor([1, 2, 3, 5, 8, 3, 1, 4, 5, 9, 4, 3, 7, 0, 7, 7, 4, 1, 5, 6, 1]), tensor([9, 0, 9, 9, 8, 7, 5, 2, 7, 9, 6, 5, 1, 6, 7, 3, 0, 3, 3, 6, 9]), tensor([9, 1, 0, 1, 1, 2, 3, 5, 8, 3, 1, 4, 5, 9, 4, 3, 7, 0, 7, 7, 4])]
